<a href="https://colab.research.google.com/github/adityab-tech/Prism/blob/main/W8_Prism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/shiyu-coder/Kronos.git

Cloning into 'Kronos'...
remote: Enumerating objects: 371, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 371 (delta 70), reused 52 (delta 52), pack-reused 265 (from 1)
Receiving objects: 100% (371/371), 9.30 MiB | 24.00 MiB/s, done.
Resolving deltas: 100% (175/175), done.


In [2]:
!git clone https://github.com/NeoQuasar/Kronos.git
%cd Kronos

fatal: destination path 'Kronos' already exists and is not an empty directory.
/content/Kronos


In [3]:
!pip install -q yfinance huggingface_hub transformers accelerate pyyaml

In [4]:
import os
import sys
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("/content/Kronos")
sys.path.append("/content/Kronos/finetune_csv")

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import finetune_csv.config_loader as cfg

sys.modules["config_loader"] = cfg

In [7]:
#dont use original config woh alibaba dataset pe train hua hai

In [13]:
config_text = """
data:
  data_path: "/content/Kronos/finetune_csv/data/reliance.csv"
  lookback_window: 30
  predict_window: 5
  max_context: 30
  clip: 5
  train_ratio: 0.6
  val_ratio: 0.2
  test_ratio: 0.2

training:
  tokenizer_epochs: 2
  basemodel_epochs: 2
  batch_size: 32
  learning_rate: 0.0001
  num_workers: 2

model_paths:
  exp_name: "Reliance"
  pretrained_tokenizer: "NeoQuasar/Kronos-Tokenizer-base"
  pretrained_predictor: "NeoQuasar/Kronos-base"
  base_save_path: "/content/Kronos/checkpoints"
  finetuned_tokenizer: "/content/Kronos/checkpoints/tokenizer/best_model"

experiment:
  train_tokenizer: true
  train_basemodel: true
  skip_existing: false

device:
  use_cuda: true
  device_id: 0
"""

config_path = "/content/Kronos/finetune_csv/config.yaml"

with open(config_path, "w") as f:
    f.write(config_text)

print("✅ Config saved at:", config_path)

✅ Config saved at: /content/Kronos/finetune_csv/config.yaml


In [15]:
from config_loader import CustomFinetuneConfig
config = CustomFinetuneConfig("/content/Kronos/finetune_csv/config.yaml")
print(config.finetuned_tokenizer_path)
print(config.lookback_window)
print(config.predict_window)

/content/Kronos/checkpoints/tokenizer/best_model
30
5


In [16]:
from model import Kronos, KronosTokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [17]:
tokenizer = KronosTokenizer.from_pretrained(
    config.finetuned_tokenizer_path
).to(device)

predictor = Kronos.from_pretrained(
    config.pretrained_predictor_path
).to(device)

tokenizer.eval()
predictor.eval()

print("Models Loaded Successfully")

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/Kronos/checkpoints/tokenizer/best_model'. Use `repo_type` argument if needed.

In [18]:
from huggingface_hub import hf_hub_download

config_file = hf_hub_download(
    repo_id="NeoQuasar/Kronos-Tokenizer-base",
    filename="config.json"
)

print(config_file)

config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

/root/.cache/huggingface/hub/models--NeoQuasar--Kronos-Tokenizer-base/snapshots/0e0117387f39004a9016484a186a908917e22426/config.json


In [12]:
print("finetuned_tokenizer_path:", config.finetuned_tokenizer_path)
print("pretrained_tokenizer_path:", config.pretrained_tokenizer_path)


finetuned_tokenizer_path: None
pretrained_tokenizer_path: NeoQuasar/Kronos-Tokenizer-base
